# MICrONS two-photon scans, opened cold

This notebook opens the store using nothing but the store itself: no ingest code and no access to the source listings. Everything below comes from the store's own metadata, and the pixels are read through it from BossDB's TIFFs.

The store references the 19 two-photon functional scans of the MICrONS mouse, as BossDB published them, with each field's position and the scan's volume times from the DANDI 000402 NWB files. The store is the private Arraylake repo `earthmover-biology/MICrONS`; you need an Arraylake login with read access (`al auth login`).

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import zarr
from arraylake import Client

REPO = "earthmover-biology/MICrONS"

repo = Client().get_repo(REPO)
root = zarr.open_group(repo.readonly_session(tag="v2").store, mode="r")

print(root.attrs["title"])
print(root.attrs["description"])
print()
print("license:", root.attrs["license"])
for citation in root.attrs["citation"]:
    print("cite:", citation)
print("export:", root.attrs["tiff_export"])

## The scans

`two-photon` holds one group per scan, named `<session>-<scan>`. Its attributes summarize every scan.

In [ ]:
scans = root["two-photon"].attrs["scans"]
for name, scan in scans.items():
    print(f"{name:>5}: {scan['fields']} fields, {scan['volumes']:,} volumes at {scan['rate_hz']} Hz")
print(f"{sum(s['fields'] for s in scans.values())} fields in {len(scans)} scans")

## One scan's fields in space

A scan group is an OME-Zarr 0.6 scene. Its coordinate system, `motor`, is the microscope's, in micrometers and seconds, and it places each field image by a translation `(t, z, y, x)`: the first volume's time, the field's depth, and the center of its first pixel. Coplanar fields sit side by side.

In [ ]:
scan = root["two-photon/4-7"]
print(scan.attrs["scan"]["coordinates"])
placement = {t["input"]["path"]: t["translation"] for t in scan.attrs["ome"]["scene"]["coordinateTransformations"]}

fig, ax = plt.subplots(figsize=(6, 6))
for path, (t, z, y, x) in placement.items():
    level = scan[f"{path}/0"]
    pixel = scan[path].attrs["annotations"]["pixel_um"]
    height, width = level.shape[2] * pixel, level.shape[3] * pixel
    ax.add_patch(plt.Rectangle((x - pixel / 2, y - pixel / 2), width, height, fill=False,
                               color=plt.cm.viridis(z / 600)))
    ax.text(x + width / 2, y + height / 2, f"{path}\n{z:g} µm", ha="center", va="center", fontsize=8)
ax.autoscale()
ax.set_aspect("equal")
ax.invert_yaxis()
ax.set_xlabel("x (µm), posterior to anterior")
ax.set_ylabel("y (µm), lateral to medial")
ax.set_title("scan 4-7: fields placed by the scene")

## One field

A field image has axes `(t, z, y, x)`, one chunk per frame. The z axis is one plane deep; the scene uses it to place the field at its depth. `volume_times`, beside the fields, holds the exact time of each volume.

In [ ]:
field = scan["field-3"]
level = field["0"]
ms = field.attrs["ome"]["multiscales"][0]
axes = [a["name"] for a in ms["coordinateSystems"][0]["axes"]]
scale = ms["datasets"][0]["coordinateTransformations"][0]["scale"]
print(ms["name"], dict(zip(axes, scale)))
print(f"shape {level.shape} {level.dtype}, chunk {level.chunks}")
print("source:", field.attrs["annotations"]["source_url"])
print("pages:", field.attrs["annotations"]["tiff_pages"])

times = scan["volume_times"][:]
frames = level[20000:20100, 0]
fig, (left, right) = plt.subplots(1, 2, figsize=(10, 6))
left.imshow(frames.mean(axis=0), cmap="gray")
left.set_title(f"mean of volumes 20000-20099 ({times[20000]:.0f}-{times[20099]:.0f} s)")
right.plot(times[20000:20100], frames[:, 200:240, 100:140].mean(axis=(1, 2)))
right.set_xlabel("time (s)")
right.set_title("mean of a 40 x 40 pixel window")

## The two exports

BossDB exported the scans twice. Tag `v1` references the first export, which declared the pixels uint16; tag `v2` references the second, which declares the same bytes int16. The few negative values that motion correction leaves read as values near 65,535 in `v1`.

In [ ]:
first = zarr.open_group(repo.readonly_session(tag="v1").store, mode="r")
old = first["two-photon/4-7/field-3/0"][20000, 0]
new = level[20000, 0]
print("v1:", old.dtype, old.min(), old.max(), "| v2:", new.dtype, new.min(), new.max())
print("same bytes:", np.array_equal(old.view(np.int16), new))
for snapshot in repo.ancestry(tag="v2"):
    print(snapshot.id, snapshot.message[:100])